# ケーススタディ

[エコノミクス](./index.ipynb) で整理した6つの観点から、いくつかのプロトコルとDAppsを評価する。

| | 戦略的行動 | スパム・シビル | ただ乗り | 限界費用 | 限界効用 | 安定化 | 評価 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Bitcoin | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | 統合のベンチマーク |
| Gitcoin | ✓ | （集権的） | （そもそも不要） | | ✓ | | 分散性を緩めてQFを導入 |
| Nouns DAO | ✓ | ✓ | | ✓ | ✓ | | NFTならではの改善の余地 |
| Terra | ✓ | ✓ | ✓ | | ✓ | ✓（失敗） | 限界費用の欠如が失敗を招いた |
| Uniswap（UNI） | ✓ | ✓ | | （弱い） | （弱い） | | 存在意義が不明瞭なトークン |

左の3列が [クリプトエコノミクス](./cryptoeconomics.ipynb)（分散性のための合意形成）、続く3列が [トークノミクス](./tokenomics.ipynb)（自律性のためのトークン価値）の観点である。評価は伊東（2026）の講義に基づく。

## Bitcoin

P2Pの電子現金システムのためにインセンティブを活用した、最初の実用的なプロトコル（→[Bitcoin](../blockchain/bitcoin.ipynb)）。

| 観点 | 対処 |
| --- | --- |
| 戦略的行動 | PoWとNakamotoコンセンサスの組み合わせ |
| スパム・シビル攻撃 | スパムにはトランザクション手数料、シビル攻撃にはPoW |
| ただ乗り | Nakamotoコンセンサスとcoinbase（最長チェーンのブロックの作成者だけが報酬を得る） |
| 限界費用 | PoWとcoinbaseの組み合わせ。半減期で時間とともに上昇する |
| 限界効用 | P2Pの電子現金、手数料の支払い手段 |
| 安定化 | 難易度調整 |

1つの仕組みに複数の役割が込められている。例えばPoWはシビル攻撃への対策であると同時にBTCの限界費用を生み、coinbaseはただ乗りへの対策であると同時にBTCの供給の仕組みでもある。合意形成とトークン価値が一体として設計されている点で、自律分散的な設計のベンチマークといえる。

残る検討課題は、スケーラビリティ、ファイナリティ、エネルギー効率といった他の規範と、プロトコル自体のガバナンス（仕様変更の決め方）である。

## Gitcoin

Ethereum上のDAppsで、オープンソースプロジェクトへの寄付のプラットフォーム。

- 支援者（マッチングファンド）がETHやステーブルコインをプールに拠出する
- 個人が各プロジェクトに直接寄付する
- プールの資金を各プロジェクトにどう配分するかを、個人の寄付額をもとに **二次の資金調達（Quadratic Funding, QF）** で決める（Buterin, Hitzig & Weyl, 2019）
- どのプロジェクトを掲載するかは、集権的な運営が決める

QFでは、プロジェクト $p$ に寄付した人 $i$ の寄付額を $c_i^p$ とすると、プロジェクトが受け取る総額を

$$
F^p = \left( \sum_i \sqrt{c_i^p} \right)^2
$$

とし、直接の寄付額 $\sum_i c_i^p$ との差額をマッチングプールから補う（プールが足りなければ比例的に縮める）。寄付の **金額** よりも **人数** が重視されるので、少額でも多くの人に支持されているプロジェクトに資金が集まる。これは二次投票（$n$ 票のコストが $n^2$）を公共財の資金調達に応用したもので、理論上は公共財の最適な供給量を実現する。

In [1]:
import numpy as np


def quadratic_funding(contributions: dict[str, list[float]], pool: float) -> dict[str, float]:
    # 各プロジェクトへのマッチング額を返す
    ideal = {p: np.sum(np.sqrt(c)) ** 2 - np.sum(c) for p, c in contributions.items()}
    scale = min(1.0, pool / sum(ideal.values()))
    return {p: m * scale for p, m in ideal.items()}


contributions = {
    "A（大口1人が100ドル）": [100.0],
    "B（10人が10ドルずつ）": [10.0] * 10,
    "C（100人が1ドルずつ）": [1.0] * 100,
}
matching = quadratic_funding(contributions, pool=5_000)
for p, m in matching.items():
    print(f"{p}: 寄付 {sum(contributions[p]):5.0f}ドル + マッチング {m:7.1f}ドル")

print()
sybil = {"A（大口1人が100ドル）": [100.0], "A'（同じ人が100アカウントに分けて1ドルずつ）": [1.0] * 100}
for p, m in quadratic_funding(sybil, pool=1e9).items():
    print(f"{p}: マッチング {m:7.1f}ドル")

A（大口1人が100ドル）: 寄付   100ドル + マッチング     0.0ドル
B（10人が10ドルずつ）: 寄付   100ドル + マッチング   416.7ドル
C（100人が1ドルずつ）: 寄付   100ドル + マッチング  4583.3ドル

A（大口1人が100ドル）: マッチング     0.0ドル
A'（同じ人が100アカウントに分けて1ドルずつ）: マッチング  9900.0ドル


直接の寄付額はどれも100ドルだが、寄付者が多いほどマッチング額が大きくなる。裏を返すと、1人が多数のアカウントに分けて寄付すればマッチング額を大きく増やせるので、QF（および二次投票）は **シビル攻撃に非常に弱い**。1人1アカウントを担保できても、複数人で示し合わせて寄付し合う **談合（collusion）** にも弱い。

| 観点 | 対処 |
| --- | --- |
| 戦略的行動 | QFで支持の広さを反映するが、シビル攻撃と談合に弱い |
| スパム・シビル攻撃 | 集権的に対処（Gitcoin Passport：SNSアカウントなどを提出させてスコア化） |
| ただ乗り | 寄付のプラットフォームなので、そもそも対処する必要がない |

Gitcoinは2021年にDAO化を目指してガバナンストークンGTCを発行した。

| 観点 | 対処 |
| --- | --- |
| 限界費用 | 全量がプレマイン（事前発行）なので、おそらくない |
| 限界効用 | ステーキングでPassportのスコアを高められる。ガバナンストークンとしての機能 |
| 安定化 | なし |

QFという新しい規範を実装するために、分散性をある程度犠牲にしている。GTCを通じてQFと自律分散性を両立できれば理想的だが、現状はその水準に達していない。

## Nouns DAO

Ethereum上のDAppsで、NFTを使ったDAO（→[DAO](../applications/dao.ipynb)）。

1. 1日に1体、Nounと呼ばれるNFTを自動的に発行する
2. NounはETHで入札するデイリーオークションにかけられ、落札代金はトレジャリーにプールされる
3. Nounの保有者は、プールされたETHの使い道を決める投票に参加できる。投票はトークンステーキング方式だが、投票の結果に応じたNounの再分配（報酬もペナルティも）はない

| 観点 | 対処 |
| --- | --- |
| 戦略的行動 | トークンステーキング |
| スパム・シビル攻撃 | トークンステーキング |
| ただ乗り | 投票しても報酬はないので、対処していない。実際にはコミュニティの熱意が投票の動機になっている |
| 限界費用 | デイリーオークションの落札額（ETH） |
| 限界効用 | アートとして＋ガバナンストークンとして |
| 安定化 | なし |

欠けているただ乗りへの対策と価値の安定化を、NFTならではの仕組みで補えないかという提案がある。

- 投票に使ったNounの特徴（パーツ）を、以降は出にくくする（レアにする）ことで投票への参加を促す。CryptoPunksのように、NFTにはレアリティという価値の概念がある
- 過去の平均落札額に応じてオークションの頻度を変え、難易度調整のように価格を安定させる

実際、2023年に導入されたfork機能では、Nounの市場価格がトレジャリーの1体あたりの額を下回ったことで裁定取引が起き、トレジャリーが大きく流出した（→[DAO](../applications/dao.ipynb)）。価値の安定化の仕組みがなかったことが表面化した例といえる。

## Terra

アルゴリズム型ステーブルコインを目指したプロトコル（→[ステーブルコイン](../applications/defi/stablecoins.ipynb)）。

- **UST**（TerraUSD）：米ドルにペッグするステーブルコイン
- **LUNA**：Terraチェーンのネイティブトークン。合意形成に参加した報酬（coinbase）として発行される
- 1 USTは常に1ドル分のLUNAと交換できる。1 UST < 1ドルのときは、USTを安く買ってLUNAに交換すれば利益が出る。交換されたUSTはバーンされて供給が減り、USTの価格が戻る（逆の場合も同様）
- 合意形成には、PoS（LUNA）、Nakamotoコンセンサス、coinbase（LUNA）を使う

| 観点 | 対処 |
| --- | --- |
| 戦略的行動 | PoSとNakamotoコンセンサス |
| スパム・シビル攻撃 | スパムにはトランザクション手数料、シビル攻撃にはPoS |
| ただ乗り | PoSとNakamotoコンセンサス |
| 限界費用（LUNA） | **なし**。LUNAはUSTとの交換やPoSの報酬として自動的に新規発行される |
| 限界効用（LUNA） | ステーキングで手数料（UST）とcoinbase（LUNA）を得られる |
| 安定化 | UST–LUNAの交換、coinbaseやLUNAのバーン率の調整 |

Bitcoinを参考にしていて、一見よくできている。しかし2022年5月、USTはペッグを維持できずに崩壊した。

```mermaid
flowchart LR
    a["UST < 1ドル"] --> b["USTをLUNAに<br/>交換する需要が増える"] --> c["LUNAが大量に<br/>新規発行される"] --> d["LUNAの価格が下がる"] --> e["1 USTを1ドル分の<br/>LUNAに替えても<br/>売れなくなる"] --> a
```

参加者の期待に注目した実証研究がいくつかある（Uhlig, 2022; Briola et al., 2023）。設計の観点から見ると、根本的な原因は **ペッグを支えるLUNAに限界費用がないこと** と考えられる。

通常、ステーキングされたトークンは他の用途に使えなくなるので機会費用が生じ、それが限界費用になる。しかしLUNAは用途そのものがステーキングなので、ステーキングしても機会費用が生じない。「ステーキングするために価値があり、価値があるからステーキングする」という同語反復的な構造になっていて、価格の下支えがなかった。

## Uniswap（UNI）

Ethereum上の代表的なDEX（→[AMM](../applications/defi/amm.ipynb)）。2020年に、流動性提供者を含む過去の利用者などにガバナンストークンUNIを配布した。ここではAMMの仕組みではなくUNIについて考える。

| 観点 | 対処 |
| --- | --- |
| 戦略的行動 | トークンステーキング（投票） |
| スパム・シビル攻撃 | トークンステーキング（提案に必要なUNIの量） |
| ただ乗り | 投票しても報酬はないので、対処していない。投票率が低いことが指摘されている（Barbereau et al., 2022, 2023） |
| 限界費用 | 流動性提供の機会費用からLPトークンの報酬を引いたもの（弱い） |
| 限界効用 | ガバナンストークンとして（弱い） |
| 安定化 | なし |

AMMの仕組みは非常に美しい一方で、UNIにはただ乗りの問題が残り、限界費用・限界効用ともに弱い。

AMMの手数料をLPトークンの保有者だけでなくUNIの保有者にも分配すれば限界効用は高まる。これは **フィースイッチ** としてUniswapのコミュニティで長く議論されてきた。しかしそれならば、LPトークンそのものをガバナンストークンにすればよいのではないか、という疑問が残り、UNIの存在意義は不明瞭である。

:::{admonition} コラム：UNIはなぜ生まれたか
:class: note

Uniswapは2018年に稼働し、当初はトークンを持たなかった。2020年、UniswapのコードをコピーしたSushiSwapが登場し、Uniswapのプールに流動性を提供していた人にガバナンストークンSUSHIを配るというインセンティブで、Uniswapから大量の流動性を奪った（**ヴァンパイアアタック**）。これに対抗するために、Uniswapも存在意義の不明瞭なガバナンストークンUNIを配布することになった（Fan et al., 2023）。

トークンが設計上の必然からではなく、競争上の必要から生まれた例である。
:::

## まとめ

- Bitcoinは、1つの仕組みが合意形成とトークン価値の両方の役割を担う、統合のベンチマークである
- Gitcoinは新しい規範（QF）を優先して分散性を犠牲にし、Nouns DAOとUniswapはただ乗りや価値の安定化を扱っていない
- Terraは観点をほぼ網羅しているように見えたが、限界費用の欠如が崩壊を招いた

6つの観点を **同時に** 満たす設計がいかに難しいかが分かる。

## 参考文献

- 伊東謙介 (2026). 「エコノミクス入門その2」東京大学ブロックチェーンイノベーション寄付講座 ブロックチェーン公開講座2026 第18回.
- Ito, K. (2024). Cryptoeconomics and Tokenomics as Economics: A Survey with Opinions. In *International Conference on Blockchain 2024*.
- Buterin, V., Hitzig, Z. & Weyl, E. G. (2019). [A Flexible Design for Funding Public Goods](https://doi.org/10.1287/mnsc.2019.3337). *Management Science*, 65(11), 5171–5187.
- Uhlig, H. (2022). [A Luna-tic Stablecoin Crash](https://www.nber.org/papers/w30256). NBER Working Paper 30256.
- Briola, A., Vidal-Tomás, D., Wang, Y. & Aste, T. (2023). Anatomy of a Stablecoin's Failure: The Terra-Luna Case. *Finance Research Letters*, 51.
- Barbereau, T. et al. (2022). [DeFi, Not So Decentralized: The Measured Distribution of Voting Rights](https://doi.org/10.24251/HICSS.2022.741). *HICSS 2022*.
- Fan, S. et al. (2023). Towards Understanding Governance Tokens in Liquidity Mining: A Case Study of Decentralized Exchanges. *World Wide Web*, 26.